# Study 952 — After-Tax Equivalent 🏛️

**At what tax bracket do municipal bonds actually start beating taxable credit?**

Municipal bond coupons escape federal income tax. Every brochure turns that into the same
promise: above some marginal rate, munis are the better bond. This study measures the rate.

We pull each fund's daily closes **twice** — total return and price only — and recover the
monthly **income** leg as the difference, because income is the only leg the tax code
touches. Then we tax it: munis exempt from federal tax and the 3.8% NIIT surtax, taxable
credit paying `federal + NIIT + state`, T-bills state-exempt, the price leg left untaxed
(a buy-and-hold assumption we sweep). The after-tax difference turns out to be **exactly
linear in the bracket**, so the **break-even rate** solves in closed form.

Tape: **MUB, VTEB, SUB, HYD** (munis) against **AGG, LQD, VCIT** (taxable credit) and
**BIL** (cash), 2004-02 → 2026-06, 269 months.

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `0f379866a58d`); the only
live cells run the offline synthetic control. As-of 2026-06-30.*


## 1. Why a smaller coupon can still be the bigger cheque

Over the 199 months both funds existed, **MUB** (national investment-grade munis) paid **2.66%/yr** of income while **VCIT** (intermediate investment-grade corporate bonds, a similar maturity) paid **3.63%/yr**. The muni yields less — by design. The question is whether the tax break more than closes that 0.97 pp gap, and at what bracket.

In [1]:
R = {'inc_mub': 2.66, 'inc_vcit': 3.63}
gap = R['inc_vcit'] - R['inc_mub']
print('MUB  (muni)  income %.2f%%/yr  - federally tax-free' % R['inc_mub'])
print('VCIT (corp)  income %.2f%%/yr  - fully taxable' % R['inc_vcit'])
print('pre-tax yield give-up: %.2f pp' % gap)
print('so the muni needs a tax rate of about %.1f%% just to draw level on income'
      % (100 * gap / R['inc_vcit']))

MUB  (muni)  income 2.66%/yr  - federally tax-free
VCIT (corp)  income 3.63%/yr  - fully taxable
pre-tax yield give-up: 0.97 pp
so the muni needs a tax rate of about 26.7% just to draw level on income


> 🔬 **For the quants.** The income leg is not quoted anywhere — we reconstruct it as `total return − price return` from two separate `yfinance` pulls (`auto_adjust` on and off). Ten of 199 months come out very slightly negative, an artefact of ex-dividend dates straddling a month end; flooring them at zero moves the mean by less than 0.001 bps/month and changes no conclusion.

## 2. The bracket where the two lines cross — and how blurry that line is

Because only the income leg is taxed, the after-tax difference is a straight line in the tax rate. Solve for where it hits zero and you get the **break-even bracket**. There are two honest versions of that number, and they are not the same question.

**(a) Compare the coupons.** How high must your rate be before the muni's realised *income stream* is worth more than the corporate one? Distribution streams are smooth, so this is measured tightly:

| Muni | vs taxable | Income-leg break-even | 95% range |
|---|---|--:|--:|
| MUB | VCIT | **26.7%** | [23.4%, 29.7%] |
| MUB | LQD | **28.9%** | [27.1%, 30.7%] |
| VTEB | VCIT | **34.3%** | [31.0%, 38.2%] |

Call it **27–34%**, give or take three points. That is below the top two US brackets (35.8% and 40.8% with the surtax), and it is this study's one solid result.

**(b) Compare the whole return.** Do the same thing on *total* returns, so the price legs count too, and the point estimate rises to **35.0%** (MUB/VCIT). Tempting — but resample the tape and the honest range is **[-10.9%, 81.5%]**. 41% of resamples say *no US bracket is high enough*; 6% say the muni already wins with no tax at all. The extra width is entirely the price legs, whose difference is -2.53 bps/month at *t* = -0.36 — i.e. nothing.

So the difference between (a) and (b) is not a finding. It is noise wearing a decimal point.

## 3. The trap: why a *t*-stat here is not what it looks like

Race MUB against **AGG** (the US Aggregate index fund) and the muni wins by **+11.91 bps/month** at the top bracket — *t* = **+2.22**, the one result on our tape that looks statistically solid. It is not.

The after-tax gap is `pre-tax gap + tax break`. The **tax break is a coupon stream**: big and almost the same every month. The **pre-tax gap** is the difference of two bond-fund returns: small and wildly noisy. Add a near-constant to a noisy series and the average jumps while the wobble does not — so the *t*-stat rises **by arithmetic**, with no new information anywhere.

For MUB vs AGG the tax break supplies **85.5%** of the average and **0.30%** of the wobble. Strip it out and the pre-tax gap is +1.73 bps/month at *t* = +0.31: nothing. The same trick works on our synthetic *twin* world, where two identical bonds cross *t* = 2 as the bracket rises with nothing planted at all.

**Every** row on this tape that clears the significance bar clears it this way. A *t*-stat built from a tax constant tests whether the tax code is nonzero. It is.

## 4. So what happens above the break-even?

Almost nothing you could measure. At a 40.8% effective rate MUB beats VCIT by **+0.21 pp/yr** — a *t* of **+0.25**, with a bootstrap range of [-12.5, +15.6] bps/month that comfortably contains zero. Split the sample at 2017 and the sign even flips (-2.40 then +4.73 bps/month). The honest reading is that the muni market has **priced the tax break away** to about the level of the top brackets, leaving a dead heat for the people it was supposedly free money for.

## 5. What the tax knobs are worth (all of them are assumptions)

None of the tax rates is data — they are imposed. So we swept them all. Adding a California-scale state tax (13.3%) moves the MUB-vs-VCIT edge from +1.72 to +2.47 bps/month, because a *national* muni fund's income is mostly out-of-state and gets taxed by your state too. Only a **single-state** fund helps meaningfully (+4.53 bps/month, still *t* = +0.65). Taxing the price leg at 23.8% instead of leaving it unrealised moves it to +2.32. No knob rescues the comparison.

## 6. Live check — the machinery is honest (offline synthetic)

We build a fake bond world where we *know* the answer: the taxable leg is planted 150 bp richer pre-tax, so the true break-even is exactly 33.3%. The solver has to find it from the tape alone. Then we plant twins — identical yields — and check the solver reports a break-even of zero and no pre-tax difference.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from after_tax import data, strategy as st
for ss, label in [(1.0, 'planted 150 bp gap'), (0.0, 'twins (the null)')]:
    panel, truth = data.synthetic_panel(signal_strength=ss, seed=952)
    d = st.synthetic_detect(panel)
    print('%-20s true break-even %6.2f%%  ->  recovered %6.2f%%   pre-tax diff t=%+.2f'
          % (label, truth['planted_breakeven']*100, d['breakeven']*100, d['pretax_t']))

planted 150 bp gap   true break-even  33.33%  ->  recovered  38.97%   pre-tax diff t=-8.29


twins (the null)     true break-even   0.00%  ->  recovered   8.05%   pre-tax diff t=-1.46


## Verdict

- **Signal — Weak.** One number here is solid and it is not a win: on realised coupons the muni needs about **26.7%** ([23.4%, 29.7%]) against like-for-like corporate credit — below the top brackets. Everything past that dissolves: on *total* returns the crossover could be anywhere from -11% to 82%; the top-bracket edge is +0.21 pp/yr at *t* = +0.25; and every significant-looking *t* on this page is the tax constant, not a market.
- **Tradability — Fragile.** The asset-location call is real and costs essentially nothing (one round trip, 0.03 bps/month amortised): in a top bracket, hold the muni fund in the taxable account. But the total-return margin is inside the noise, the answer moves with a bracket you *assume*, and as a long-short trade a mere 25 bps/yr of borrow turns it negative.
- **The plain-words version.** The tax break on munis is real, and the market has already priced it in to roughly the level of the top US brackets. If you are in one of those brackets, hold munis in your taxable account — the coupon comparison says so clearly. Just do not expect the tape to pay you for noticing: expect a tie.